# 04 — FIDC credit

**The question:** *How delinquent is this receivables fund's book, how
concentrated is it, and where does the credit actually come from?*

A FIDC (*fundo de investimento em direitos creditórios*) buys receivables.
Three things decide whether it is healthy: how much of the book is late, how
much of it depends on one debtor, and who is selling it the paper.

This notebook is also the one with a **regime break** sitting in the middle of
the obvious window. Getting that wrong turns a format change into a credit
event, so it is handled first and explicitly.

Endpoints: `panel`, `fidc_portfolio`, `fidc_sacados`, `fidc_cedentes`.

In [ ]:
# The SDK is not on PyPI. From the repository root:
#
#     pip install -e sdk/
#
# Auth is the shared publishable key printed in the docs. It is for TESTING:
# everyone reading the docs has the same one, so it identifies the project and
# not you. It puts you on the ANONYMOUS tier. Set SILO_TOKEN to a GitHub
# sign-in token (notebook 00) to run signed in.
import os

os.environ.setdefault("SILO_URL", "https://zcjbtpxuhdekpwcxmepn.supabase.co")
os.environ.setdefault(
    "SILO_ANON_KEY", "sb_publishable__yfFQsykAglrvc9GS6_PYw_B24ex437"
)

import pandas as pd

from silo_client import SiloClient

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

silo = SiloClient()
print(f"tier            : {silo.tier}")
print(f"catalog version : {silo.catalog()['version']}")

In [ ]:
def as_of(*datasets: str) -> pd.DataFrame:
    """Print how fresh every dataset this notebook relies on actually is.

    Run this FIRST, every time. A stale warehouse then shows up in the output
    instead of being silently baked into a number further down.

      as_of            the newest period that has landed AND has elapsed.
                       This is freshness.
      complete_through the newest period classified COMPLETE — what the
                       default windows serve.
      newest_period    the newest period KEY. It can sit in the FUTURE when a
                       family files forward-dated (FIP is keyed 31-December).
                       Never read this as freshness.
      landed_at        when ingest last SUCCEEDED. A later failed run never
                       advances it.
      notes            a caveat the dates cannot carry. Printed in full below,
                       never summarised, never dropped.
    """
    cov = pd.DataFrame(silo.coverage())
    rows = cov[cov["dataset"].isin(datasets)].copy()
    missing = set(datasets) - set(rows["dataset"])
    if missing:
        raise RuntimeError(f"coverage() has no row for {sorted(missing)}")
    print(
        rows[
            ["dataset", "as_of", "complete_through", "newest_period", "landed_at"]
        ].to_string(index=False)
    )
    for r in rows.itertuples():
        if r.notes:
            print(f"\n  CAVEAT [{r.dataset}]\n  {r.notes}")
    return rows.set_index("dataset")

In [ ]:
COVERAGE = as_of("funds_fidc", "fidc_sectors", "fidc_sacados",
                 "fidc_cedentes", "fidc_scr")

## Read that `funds_fidc` caveat again

It is the whole reason this notebook exists. The machine-readable form is
`catalog().regime_breaks`, and it says the same thing in more detail:

In [ ]:
for brk in silo.catalog()["regime_breaks"]:
    print(f"dataset  : {brk['dataset']}")
    print(f"column   : {brk['column']}")
    print(f"boundary : {brk['boundary']}")
    print(f"before   : {brk['before']}")
    print(f"after    : {brk['after']}")
    print(f"NEVER    : {brk['never']}")

And `metric_coverage` shows the consequence in rows. `first_period` for
`(fidc, delinquency)` is **2025-01-31**, and `filed_rows` is a fraction of
`total_rows` — not because reporting is patchy, but because the column did not
exist in the source before that date.

In [ ]:
mc = pd.DataFrame(silo.metric_coverage())
fidc_mc = mc[mc["entity_type"] == "fidc"].copy()
fidc_mc["filed_pct"] = (100 * fidc_mc["filed_rows"] / fidc_mc["total_rows"]).round(1)
print(fidc_mc.to_string(index=False))

## Pick a fund

Anonymous callers enumerate the family from the `funds` **view** — the one
surface that pages with `limit`/`offset`. (Signed in, `panel` universe mode does
a whole family in one paged call; see notebook 00 for the tier table.)

In [ ]:
universe = pd.DataFrame(silo.view(
    "funds", entity_type="eq.fidc", order="n_reports.desc",
    select="cnpj,fund_name,status,first_period,last_period,n_reports",
    limit=10))
universe.head(10)

In [ ]:
CNPJ = "09221411000181"   # VERTIGO FIDC MULTISEGMENTOS — files since 2019-01
name = universe.set_index("cnpj").loc[CNPJ, "fund_name"]
print(name)

## The break, seen in the data

Ask for `delinquency` and `nav` across the boundary. The panel does not return
a zero for the months before 2025-01 — it returns **no row at all**, because the
derived arm filters nulls before emitting. Pivot locally and it becomes NaN,
which is exactly what it is.

In [ ]:
across = silo.panel([CNPJ], ["nav", "delinquency"], freq="month",
                    start="2024-06-01", end="2025-06-30", wide=True)
across.columns = across.columns.droplevel(0)
across

In [ ]:
BOUNDARY = pd.Timestamp("2025-01-01")   # panel keys months on the FIRST of the month

before = across.loc[across.index < BOUNDARY, "delinquency"]
after = across.loc[across.index >= BOUNDARY, "delinquency"]

print(f"months before 2025-01 : {len(before):>2}   non-null delinquency: {before.notna().sum()}")
print(f"months from  2025-01  : {len(after):>2}   non-null delinquency: {after.notna().sum()}")
print()
print("Those NaNs are NOT zero, NOT clean books, and NOT a missing month.")
print("CVM's pre-2025 FIDC file (tab II/III) carries no delinquency field at all.")

### What chain-linking through it would produce

This cell computes the wrong answer **on purpose**, so the size of the mistake
is visible rather than theoretical. Nothing downstream uses it.

In [ ]:
naive = across["delinquency"].fillna(0)          # <- the mistake
print("if you fillna(0) across the boundary, December 2024 -> January 2025 reads:")
print(f"  2024-12 delinquency : R$ {naive.loc['2024-12-01']:>14,.2f}   (INVENTED)")
print(f"  2025-01 delinquency : R$ {naive.loc['2025-01-01']:>14,.2f}")
print(f"  'change'            : +{100 * float('inf') if naive.loc['2024-12-01'] == 0 else 0:.0f}%"
      "  — an infinite deterioration, out of a format change")
print()
print("The series STARTS at 2025-01. It does not step up at 2025-01.")

del naive

## The delinquency rate — a local division

`delinquency` is the delinquent portfolio in **BRL**. It is not a rate. The API
does not serve the ratio, because it does not serve anything it did not receive
from a filing — so ask for both ingredients in one call and divide here, where
the choice of denominator is visible.

`nav` is one reasonable denominator (net assets). `receivables` — the tab II
total — is another, and arguably the better one for a credit question. They are
different numbers and they answer different questions.

In [ ]:
START = "2025-01-01"                       # the series begins here. Not earlier.

p = silo.panel([CNPJ], ["nav", "delinquency", "receivables"],
               freq="month", start=START, wide=True)
p.columns = p.columns.droplevel(0)

p["delinq_over_nav_pct"] = 100 * p["delinquency"] / p["nav"]
p["delinq_over_receivables_pct"] = 100 * p["delinquency"] / p["receivables"]

print(f"CAVEAT: FIDC delinquency is filed only from 2025-01-31 "
      f"(catalog().regime_breaks).")
print(f"        This series starts there and is never chain-linked across it.\n")
p[["nav", "receivables", "delinquency",
   "delinq_over_nav_pct", "delinq_over_receivables_pct"]].tail(12)

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(p.index, p["delinq_over_receivables_pct"], marker="o", markersize=3,
        label="delinquency / receivables")
ax.plot(p.index, p["delinq_over_nav_pct"], marker="o", markersize=3,
        label="delinquency / NAV")
ax.axvline(pd.Timestamp(START), color="crimson", linestyle="--", linewidth=1)
ax.annotate("series STARTS here\n(2025-01 regime break)",
            xy=(pd.Timestamp(START), ax.get_ylim()[1] * 0.92),
            fontsize=8, color="crimson")
ax.set_title(f"{name[:52]}\ndelinquency, as filed — no data before 2025-01")
ax.set_ylabel("%")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print("The line does not extend left of the dashed rule. That is not a chart")
print("bug and not a fund that was clean until 2025 — it is where the column")
print("starts existing.")

## The receivables book is a hierarchy

`fidc_portfolio(kind="sector")` serves informe tab II as one row per code:

* `TOTAL` — the whole receivables book.
* a **lettered** code (`A`..`K`) — a sector.
* a code with a **digit** (`C1`, `F3`) — a member of its lettered parent, named
  in `parent`.

**Sum leaves or sum parents, never both.** Not every letter has children, so
"sum the numbered rows" is not the same as "sum the book".

In [ ]:
MONTH = "2026-07-31"          # fidc_* functions key on CVM's filed month-END

book = pd.DataFrame(silo.fidc_portfolio(CNPJ, kind="sector",
                                        start=MONTH, end=MONTH, limit=200))
print(f"{len(book)} rows filed; the non-zero ones:\n")
print(book[book["value"] > 0].to_string(index=False,
                                        float_format=lambda v: f"{v:,.2f}"))

In [ ]:
total = book.loc[book["code"] == "TOTAL", "value"].iloc[0]
parents = book[(book["code"] != "TOTAL") & (book["parent"].isna())]["value"].sum()
children = book[book["parent"].notna()]["value"].sum()

print(f"TOTAL (tab II carteira)      : R$ {total:>16,.2f}")
print(f"sum of lettered sectors A..K : R$ {parents:>16,.2f}   <- equals TOTAL")
print(f"sum of numbered members      : R$ {children:>16,.2f}   <- a SUBSET")
print(f"parents + children           : R$ {parents + children:>16,.2f}   <- DOUBLE COUNTS")
print()
print(f"letters with no children: "
      f"{sorted(set(book[book['parent'].isna()]['code']) - set(book['parent'].dropna()) - {'TOTAL'})}")
print("Those carry their value at the letter, so summing only the numbered rows")
print("silently drops them.")

### The same book, graded by BACEN

`kind="scr_debtor"` and `kind="scr_operation"` are the SCR grade ladders
`AA`..`H` for the **same receivables**, graded by debtor and by operation
respectively. Two views of one book, not two books — never add them together.

Tab X exists **from 2023-10 only**. A month before that has no SCR rows at all;
that is not a fund with ungraded paper.

And a filed `0` is not the same as no row. The fund below files the whole ladder
as zero while tab II reports a real receivables balance — a disagreement between
two of CVM's own tabs, served as filed. It is not reconciled here or upstream,
and it is worth noticing rather than averaging away.

In [ ]:
scr = pd.DataFrame(silo.fidc_portfolio(CNPJ, kind="scr_debtor",
                                       start=MONTH, end=MONTH, limit=100))
SCR_CNPJ = CNPJ

if scr.empty:
    print(f"{CNPJ} filed NO tab X rows for {MONTH}.")
    print("Two distinct reasons produce an empty result here, and they are not")
    print("the same thing: the month predates 2023-10 (tab X did not exist), or")
    print("this fund simply did not file it. Neither is 'ungraded paper'.\n")
    for cand in universe["cnpj"]:
        rows = silo.fidc_portfolio(cand, kind="scr_debtor",
                                   start=MONTH, end=MONTH, limit=100)
        if rows:
            SCR_CNPJ, scr = cand, pd.DataFrame(rows)
            print(f"showing {SCR_CNPJ}, which did file it:\n")
            break

if scr.empty:
    print("no fund in this sample filed tab X for this month")
else:
    graded = scr[scr["value"] > 0][["code", "item", "value"]]
    print(f"SCR ladder by DEBTOR, {SCR_CNPJ}, {MONTH}:")
    print(graded.to_string(index=False, float_format=lambda v: f"{v:,.2f}")
          if len(graded) else "  every grade filed as 0.00")
    print()
    print(f"graded total (tab X) : R$ {scr['value'].sum():>16,.2f}")
    if SCR_CNPJ == CNPJ:
        print(f"receivables  (tab II) : R$ {total:>16,.2f}")
        print()
        print("CAVEAT: these are two filings of ONE book. A difference between")
        print("them is CVM's, served as filed — not reconciled here or upstream.")
    print()
    print("scr_debtor and scr_operation grade the SAME receivables two ways.")
    print("They are two views of one book. Never add them together.")

## Concentration: who owes the fund

Tab VIII publishes the **25 largest debtors as `(rank, value)` and nothing
else** — CVM anonymizes them, so there is no debtor identity anywhere in this
data and no debtor-side lookup exists.

`seq` is CVM's rank **as filed** and is never recomputed from `valor`: 65 of
3,043 funds filed a non-descending series in 2026-07, and they are served that
way rather than tidied.

In [ ]:
sac = pd.DataFrame(silo.fidc_sacados(CNPJ, start=MONTH, end=MONTH, limit=50))
print(f"{len(sac)} ranks filed (at most 25; a fund that files fewer has fewer rows)\n")
print(sac[["seq", "valor"]].to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
print()
descending = sac["valor"].is_monotonic_decreasing
print(f"filed values monotonically decreasing? {descending}")
if not descending:
    print("-> filed out of order by the fund. Served as filed, never re-sorted.")

### The concentration ratio, and why it can exceed 1

`sacado_top1 / receivables` is a **notebook division**, not a served number.
And it can come out above 100 %: tab VIII and tab II do not share a base for
every fund. Measured across the family in 2026-07, the top-25 sum exceeded the
receivables total for 1.9 % of funds, and rank 1 alone for 0.5 %.

Those are served as filed and **never capped**. A ratio above 1 is a signal to
check the filing, not a number to clip to 1.

In [ ]:
conc = silo.panel([CNPJ], ["receivables", "sacado_top1", "sacado_top25"],
                  freq="month", start=START, wide=True)
conc.columns = conc.columns.droplevel(0)

conc["top1_pct"] = 100 * conc["sacado_top1"] / conc["receivables"]
conc["top25_pct"] = 100 * conc["sacado_top25"] / conc["receivables"]

over = ((conc["top1_pct"] > 100) | (conc["top25_pct"] > 100)).sum()
print(f"months where a concentration ratio exceeds 100%: {over}")
print("(if > 0: tab VIII and tab II disagree on the base for this fund-month;")
print(" served as filed, never capped)\n")
conc[["receivables", "sacado_top1", "sacado_top25", "top1_pct", "top25_pct"]].tail(8)

`sacado_top25` sums the ranks the fund **actually filed**, which may be fewer
than 25. Nothing is imputed for the missing ranks, so a fund filing 9 ranks and
a fund filing 25 are not directly comparable on that metric.

## Who sells the fund its paper

Tab I publishes, per fund and month, the nine largest **cedentes** (originators)
of each block, by their own CPF/CNPJ:

* **bloco A** — receivables acquired **with** substantial retention of risks and
  benefits by the originator.
* **bloco B** — **without**.

`share_pct` is the cedente's share **of that block**, never of the fund. The
block totals are not served (tab I's asset lines are not ingested), so a share
cannot be turned into reais here.

In [ ]:
# Not every FIDC files tab I slots. Find one in the family that does.
CED_CNPJ = None
for cand in universe["cnpj"].tolist() + ["21917325000103"]:
    rows = silo.fidc_cedentes(cnpj=cand, start=MONTH, end=MONTH, limit=30)
    if rows:
        CED_CNPJ, ced = cand, pd.DataFrame(rows)
        break

if CED_CNPJ is None:
    print(f"no fund in this sample filed tab I cedente slots for {MONTH}")
else:
    print(f"cedentes for {CED_CNPJ}\n")
    show = ced.copy()
    show["cedente_tickers"] = show["cedente_tickers"].map(
        lambda v: ", ".join(v) if isinstance(v, list) else v)
    print(show[["bloco", "seq", "cedente_id", "cedente_tickers", "share_pct"]]
          .to_string(index=False))

In [ ]:
if CED_CNPJ is not None:
    print("CAVEATS, all from catalog().constraints and coverage().notes:\n")
    print("1. share_pct is a percent of its BLOCK (A or B), not of the fund.")
    print("   The block totals are not served, so it cannot be turned into reais.")
    print()
    print("2. share_pct is AS FILED and dirty in the way CVM's percentage fields")
    print("   are: 9% of slots carry a value above 100 (max 19,771 in 2026-07).")
    bad = ced[(ced['share_pct'] > 100) | (ced['share_pct'] < 0)]
    print(f"   in this fund-month: {len(bad)} of {len(ced)} slots out of range")
    print()
    print("3. cedente_id is the originator's OWN filed CPF/CNPJ, kept only when")
    print("   its check digits verify. Placeholders (all-zero, all-nine) and")
    print("   unrecoverable identifiers were DROPPED at ingest, never coerced.")
    print()
    print("4. cedente_tickers is the FCA map's active listings for that CNPJ —")
    print("   a published mapping. None means NOT LISTED, not 'unknown'.")
    print()
    print("5. Slots exist from 2019-11. Nothing is matched by name, ever.")

## What this notebook did not claim

* It did not extend the delinquency series before 2025-01, in any form.
* It did not fill a single missing value.
* It did not cap a concentration ratio above 1, or re-sort a `seq` that CVM
  filed out of order.
* It did not turn a `share_pct` into reais, because the denominator is not
  published.
* It did not add the sector hierarchy's parents to its children.

## Where this goes next

* Notebook `08` looks at credit stress from the market side: who is short, and
  who is lending them the stock.
* To rank the **whole family** by how much delinquency worsened, sign in and use
  `panel` universe mode:
  `panel_all(None, ["delinquency", "nav"], entity_type="fidc", start="2025-01-01",
  min_nav=10_000_000, min_months=12)` — then rank in the notebook, requiring a
  minimum number of non-null months so a fund with two observations cannot top
  the list.

---

## The rules this notebook obeyed

* **Nothing was filled.** No forward-fill, no interpolation, no carried-forward
  last observation. A gap in a chart is a gap in the filings.
* **Every caveat was printed beside its number** — `coverage().notes`,
  `catalog().regime_breaks`, `catalog().applicability`, `float_basis` — rather
  than left in a docstring somewhere.
* **Freshness came from `coverage()`**, called before anything was claimed.

The contract these rules come from is
[Conventions & limits](https://octo-98895abd.mintlify.site/api-docs/conventions),
and its machine-readable twin is `POST /rpc/catalog`.